# ML Masterclass 07: System Design & Architecture

Welcome to Notebook 07. We are officially leaving the Jupyter environment and entering the Enterprise.

A 99% accurate model is completely useless if it takes 5 seconds to run and your website requires a 100ms response time. Machine Learning System Design is the art of balancing prediction latency, throughput, infrastructure cost, and model freshness.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"When designing an Uber-Eats ETA predictor, should you use one single massive Monolithic model for the whole world, or 10,000 separate localized models (one for each city)? What are the exact architectural trade-offs regarding pipeline orchestration vs accuracy?"*

---

## Table of Contents
1. **The Architecture Spectrum:** Batch vs. Real-Time vs. Streaming Inference.
2. **Training-Serving Skew:** The silent killer of ML systems.
3. **Feature Stores:** Solving the latency/skew dilemma.
4. **System Architecture Design:** Visualizing a Real-Time Recommendation Feed.


## 1. The Architecture Spectrum (Inference Latency)

How fast do you need the prediction?

### A. Offline Batch Inference (The Cheapest)
*   **Use Case:** Netflix generating your "Recommended Movies" row for tomorrow.
*   **Architecture:** Every night at 2 AM, an Airflow DAG spins up a massive Spark/Ray cluster, runs inference for all 100M users, and writes the results to a fast Key-Value database (Redis/DynamoDB).
*   **Pros:** Cheap, high throughput, zero user-facing latency.
*   **Cons:** Stale. If you watch a movie at 9 AM, your recommendations won't update until tomorrow.

### B. Real-Time Online Inference (The Standard API)
*   **Use Case:** Uber predicting ETA when you request a ride.
*   **Architecture:** An HTTP REST API (FastAPI) wrapped around a model loaded in RAM (via ONNX/TensorRT). When the user clicks "Request", a JSON payload is sent, features are aggregated, and the model predicts instantly.
*   **Pros:** Reacts to current conditions instantly.
*   **Cons:** Expensive (servers must be on 24/7). Highly vulnerable to Latency constraints.

### C. Streaming Inference (The Complex)
*   **Use Case:** Credit card fraud detection happening mid-swipe.
*   **Architecture:** Data flows continuously through Apache Kafka. A Flink/Spark Streaming job consumes the event, applies the model, and emits an "Approve/Deny" event back into Kafka entirely asynchronously.
*   **Pros:** Ultra-low latency, handles massive burst spikes smoothly via queue buffering.
*   **Cons:** Asynchronous logic is notoriously difficult to debug and synchronize with UI endpoints.

## 2. Training-Serving Skew

Model performance is excellent during Training/Validation. Unbelievably good. 
You deploy it. It absolutely bombs in production. Why?

**Training-Serving Skew** occurs when the data distribution or physical code used to calculate features during training is slightly different than the code used during real-time inference.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Walk through exactly how a logic bug in calculating a user's '30-day click rate' causes a model to fail.* 

**A:** 
*   **Training:** Your Data Scientist used a perfectly clean SQL query on the Data Warehouse to historically aggregate the click-rate over the 30 days *prior* to a historical event. They fed this into `model.fit()`.
*   **Serving:** Your Software Engineer needed to compute the click-rate in real-time, so they wrote a Java service querying a Redis cache. But the Java code accidentally includes clicks from *today* (which hasn't finished yet), introducing a division-by-zero bug or a slight numerical shift.
*   **Result:** The model receives a number it has never seen the distribution of. It outputs garbage predictions. No error is thrown.

## 3. The Solution: Feature Stores

To solve Training-Serving Skew and the latency of calculating features in real-time, the industry invented the **Feature Store** (e.g., Feast, Hopsworks).

A Feature Store is a centralized repository that maintains two things:
1.  **Online Store (Fast/Redis):** Houses the most recently computed features at millisecond latency for the Real-Time API to grab during inference. 
2.  **Offline Store (Slow/Parquet):** Maintains a historically pure, point-in-time correct log of those exact same features for Data Scientists to train on.

Because the Feature Store physically calculates the feature *once* and copies it to both places, Training-Serving Skew is mathematically impossible.

## 4. System Architecture: Real-Time Recommendation Feed

Below is the standard architecture diagram for a modern Machine Learning pipeline serving a real-time web application.

```mermaid
graph TD

    subgraph User Facing
        UI[Web App / Mobile Client]
        API[API Gateway / Load Balancer]
    end

    subgraph Real-Time Online Environment
        Serve[Model Inference Service (FastAPI / ONNX)]
        FS_On[(Feature Store: Online (Redis))]
    end

    subgraph Asynchronous / Streaming Environment
        Kafka[Kafka Event Stream / Clickstream]
        StreamProc[Streaming Job (Flink): Update Features]
    end

    subgraph Offline Batch Environment
        DWH[(Data Warehouse: Snowflake / BigQuery)]
        FS_Off[(Feature Store: Offline (Iceberg))]
        TrainPool[Distributed Training Cluster (Ray / GPU)]
        Reg[Model Registry (MLflow)]
    end

    %% User flow
    UI -- "GET /recommendations" --> API
    API -- "User_ID = 123" --> Serve
    
    %% Inference getting features
    Serve -- "Fetch historical features" --> FS_On
    FS_On -.-> Serve
    Serve -- "Predictions" --> API
    
    %% Data Updates
    UI -- "User Clicks" --> Kafka
    Kafka --> StreamProc
    StreamProc -- "Update User Click Rate" --> FS_On
    
    %% Backend Data Flow
    Kafka -. "Raw Logs" .-> DWH
    DWH -. "Batch Compute" .-> FS_Off
    
    %% Training Flow
    FS_Off -- "Point-in-Time Data" --> TrainPool
    TrainPool -- "Save Model Weights" --> Reg
    Reg -- "Deploy to Prod" --> Serve
```

### 🎓 DEEP QUESTION 2 ANSWERED
**Q:** *Uber-Eats: One massive global model vs 10,000 localized city models?*

**A:** 
A massive single model requires massive compute to train and struggles to learn hyper-local quirks (e.g., how rain affects traffic differently in Seattle vs Cairo). 

10,000 localized models are highly accurate for their regions, but they create a **pipeline orchestration nightmare**. You must monitor 10,000 separate accuracy metrics, handle 10,000 separate CI/CD code deployments, and deal with "cold-starts" when a city has too few orders to train a good model.

*Modern Solution:* Usually a hybrid. A massive global embedding model that learns general vehicular physics and weather impacts, paired with lightweight, localized XGBoost models that rank the final results based on local city contexts.